# 08 - Report figures (Phase 8)

**Colombo UHI practicum.** Produces the eleven publication figures for the
written report, at **300 dpi**, into `figures/report/` - plus the generated
methods and limitations sections and the data-provenance table.

`figures/` keeps the per-phase diagnostics at 150 dpi. They are NOT regenerated
here and nothing in this notebook overwrites them. Phase 8 writes a parallel,
self-consistent set.

## What this notebook needs

| Figure | Needs | Where it comes from |
|---|---|---|
| 1 decadal LST | Landsat + MODIS decadal rasters | Phase 4 export (reused) + **one new export** |
| 2 Sen's slope + FDR | trend raster | Phase 4 export (reused); FDR applied here |
| 3 valid observations | observation-count raster | **new export** |
| 4 UTFVI by epoch | six-class raster per epoch | **new export** |
| 5 Gi* / EHSA | `gi_star_gn_*.csv`, `ehsa_gn_*.csv` | committed |
| 6 SUHII series | `suhii_2000_2025.csv` | committed |
| 7 LST vs NDVI/NDBI | per-epoch driver samples | sampled here (no export task) |
| 8 GWR coefficients | `gwr_local_coefficients_gn_2020s.csv` | committed |
| 9 greening scenario | `greening_counterfactual_*.tif` | committed |
| 10 greening priority | `greening_priority_gn.csv` | committed |
| 11 data provenance | `config/params.yaml` | generated |

**Three new Earth Engine tasks**, not eleven. Everything else is either already
in `data/outputs/` or already exported to Drive by Phase 4.

> **Caveat (CLAUDE.md #1):** every temperature here is **LAND SURFACE
> TEMPERATURE**, never air temperature.
>
> **Caveat (CLAUDE.md #2):** figure 3 is the observation count, and it is a
> **product**, not a diagnostic. No other figure may be read without it.
>
> **Caveat (CLAUDE.md #3):** figure 9 is a **conditional counterfactual**, not
> a forecast. No projected land-cover map exists, because the projection did
> not validate.
>
> **The pooled-Landsat sensor step is on figure 1 on purpose.** The top row
> crosses the L5/L7/L8 changeovers, whose measured offsets over Colombo are
> several times the entire 26-year trend signal. It is drawn as a **diagnostic**
> and the bottom row - MODIS Terra, one sensor - is the comparison that is
> valid. If the two rows step together, the sensor finding is wrong and you
> need to know.

---

# Part 1 - Earth Engine

In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

# Which revision is actually on disk. Quote this if a result looks impossible.
!git --no-pager log -1 --format="HEAD %h %s (%ci)"

In [ ]:
# COLAB: RUN THIS CELL  (skip if you already ran notebook 00-07 in this runtime)
%pip install -q -r requirements.txt

In [ ]:
# COLAB: RUN THIS CELL
# Load params (single source of truth) and initialise Earth Engine.
import sys

sys.path.insert(0, os.path.abspath("src"))

# Drop any already-imported colombo_uhi modules BEFORE importing. Without this,
# re-running in a live runtime keeps the version cached in sys.modules from the
# previous run: `git pull` updates the files on disk but the import silently
# returns the OLD code.
for _name in [m for m in list(sys.modules) if m == "colombo_uhi" or m.startswith("colombo_uhi.")]:
    del sys.modules[_name]

from colombo_uhi import load_params
from colombo_uhi.auth import init_ee

params = load_params()
project = init_ee()
print("Earth Engine initialised with project:", project)
print()
for _key in ("lst_not_air_temp", "valid_obs_required", "scenario_not_forecast",
             "single_overpass", "within_epoch_only",
             "colour_is_not_the_only_channel", "figures_are_derived_not_authored"):
    print("CAVEAT:", " ".join(params["caveats"][_key].split()))
    print()

## Step 0 - prove the loaded code is current

Every name below arrived **with Phase 8**. If this checkout predates it the
guard fires here, not three cells into a render loop with a confusing
`AttributeError`.

In [ ]:
# COLAB: RUN THIS CELL
# Every import Parts 1, 2 AND 3 need, in one place.
import glob
import json
import zipfile

import ee
import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import Image, display

from colombo_uhi import (
    aoi, composites, exports, greening, prediction, reporting, spatial_stats,
    trends, uhi_metrics, viz,
)

_required = {
    "viz": [
        # plumbing
        "report_dpi", "report_figure_path", "save_report_figure",
        "report_manifest", "missing_inputs", "saturated_fraction", "panel_aspect",
        # colour-vision verification
        "simulate_cvd", "lab", "lightness", "delta_e", "palette_separation",
        "palette_lightness_profile", "is_monotonic", "is_diverging_monotonic",
        "resolve_palette", "check_palette", "cvd_report", "cvd_failures",
        "normalise_hex",
        # the new figures
        "build_decadal_lst_panel_figure", "plot_decadal_lst_panel",
        "sensor_step_banner", "stipple_coordinates",
        "build_sen_slope_stipple_figure", "plot_sen_slope_stipple",
        "build_obs_count_figure", "plot_obs_count",
        "build_utfvi_epoch_maps_figure", "plot_utfvi_epoch_maps",
        "build_lst_vs_index_facet_figure", "plot_lst_vs_index_facet",
        "build_scenario_triptych_figure", "plot_scenario_triptych",
        "build_provenance_table_figure", "plot_provenance_table",
        "spread_label_positions", "annotate_top_zones",
    ],
    "reporting": [
        "provenance_frame", "dataset_references", "scenario_report_from_raster",
        "methods_markdown", "limitations_markdown", "write_docs",
        "PROVENANCE_COLUMNS", "METHODS_PATH", "LIMITATIONS_PATH",
    ],
}
_missing = {
    name: [f for f in funcs if not hasattr(globals()[name], f)]
    for name, funcs in _required.items()
}
_missing = {name: funcs for name, funcs in _missing.items() if funcs}
if _missing:
    raise RuntimeError(
        "This checkout predates Phase 8. Missing: "
        + json.dumps(_missing, indent=2)
        + "\n\nRun `git push` locally, then re-run the clone cell above. The "
        "notebook pulls from GitHub, so uncommitted or unpushed work is "
        "invisible here."
    )

if "report" not in params:
    raise RuntimeError("params.yaml has no `report:` section; it predates Phase 8")

REPORT = params["report"]
FIGURE_DIR = REPORT["figure_dir"]
DPI = viz.report_dpi(params)
FIT_SCALE = int(params["trends"]["fit_scale_m"])
EPOCHS = list(REPORT["facet_epochs"])
DECADAL_LANDSAT_SOURCE = REPORT["decadal"]["landsat_source"]
DECADAL_MODIS_SOURCE = REPORT["decadal"]["modis_source"]
UTFVI_SCALE_M = int(params["spatial_stats"]["epoch_scale_m"])
# Resolved HERE, not in the export cell: Part 2's discovery cell needs it to
# reconstruct the exported filename, and Part 2 is routinely re-run alone after
# a runtime reconnect.
MODIS_SCALE_M = int(
    next(entry["scale_m"] for entry in params["uhi"]["suhii"]["sources"]
         if entry["key"] == DECADAL_MODIS_SOURCE)
)
OBS_SOURCE = "landsat_dry"      # the FULL archive: this is a coverage statement,
                                # not a temperature one, so pooling sensors is
                                # exactly right here.

INTERIM_DIR = "data/interim"
OUTPUT_DIR = "data/outputs"
for _directory in (INTERIM_DIR, OUTPUT_DIR, FIGURE_DIR, "docs"):
    os.makedirs(_directory, exist_ok=True)

print(f"report figures -> {FIGURE_DIR}/ at {DPI} dpi")
print(f"epochs={EPOCHS}  fit_scale={FIT_SCALE} m  "
      f"utfvi_scale={UTFVI_SCALE_M} m  modis_scale={MODIS_SCALE_M} m")
print(f"decadal rows: landsat={DECADAL_LANDSAT_SOURCE} (DIAGNOSTIC), "
      f"modis={DECADAL_MODIS_SOURCE} (valid)")
print()
print("The eleven figures:")
for _entry in viz.report_manifest(params):
    print(f"  {_entry['index']:>2}. {_entry['slug']:<22} -> {_entry['path']}")

## Step 1 - geometry and the work region

The same region Phase 4 exported into, so the reused rasters and the new ones
share a grid. `prediction.work_region` is the **GAUL** district; the GN polygons
are COD-AB, and the two differ by roughly 13 km² - which is the mismatch that
cost three Colab runs in Phase 7. Nothing here burns zones into a raster, so it
does not bite, but the numbers are printed so the discrepancy stays visible.

In [ ]:
# COLAB: RUN THIS CELL
WORK_REGION = prediction.work_region(params)
_area_km2 = WORK_REGION.area(maxError=1).divide(1e6).getInfo()
_expected = params["aoi"]["expected_areas_km2"]["district"]
print(f"work region: {_area_km2:,.1f} km2 "
      f"(COD-AB district is {_expected} km2; "
      f"{100 * (_area_km2 - _expected) / _expected:+.1f}%)")
print()
print("That gap is the GAUL-vs-COD-AB boundary mismatch recorded in")
print("docs/limitations.md. It is expected. It matters only where a product is")
print("clipped to one source and zones are burned from the other.")

## Step 2 - submit the three new exports

Everything else is reused. These three are what Phase 8 adds:

1. **MODIS Terra decadal means** - figure 1's bottom row, the one-sensor
   comparison. `trends.decadal_means` emits `mean_`, `sd_` and `n_years_` per
   window, and both Landsat and MODIS composites carry `LST_C`, so the default
   band is correct for either.
2. **Valid-observation count** - figure 3. Summed over the whole dry-season
   series, so it answers "how many usable scenes does this pixel rest on across
   the study period", which is what `caveats.valid_obs_required` is about.
3. **UTFVI six-class images** - figure 4, one band per epoch.

Each is a batch task. Expect roughly 5-15 minutes for the three together.

In [ ]:
# COLAB: RUN THIS CELL
TASKS = []

# --- 1. MODIS Terra day decadal means -------------------------------------
_modis_decadal = trends.decadal_means(
    DECADAL_MODIS_SOURCE, params, region=WORK_REGION
)
TASKS.append(exports.image_to_drive(
    _modis_decadal, product="lst_decadal", aoi="district", params=params,
    region=WORK_REGION.bounds(),
    band_order=trends.decadal_band_order(params),
    scale_m=MODIS_SCALE_M, suffix=DECADAL_MODIS_SOURCE,
))
print("submitted:", TASKS[-1].status().get("description"))

In [ ]:
# COLAB: RUN THIS CELL
# --- 2. total valid dry-season observations per pixel ---------------------
# CLAUDE.md caveat 2 in raster form. Pooling sensors is right here: this counts
# USABLE SCENES, which is a property of the archive and the cloud, not of any
# sensor's calibration.
_scenes = uhi_metrics.source_collection(OBS_SOURCE, params, region=WORK_REGION)
_dry = composites.dry_season_composites(_scenes, params)
_obs_band = params["composites"]["obs_count_band"]
_total_obs = (
    _dry.select([_obs_band]).sum().rename(_obs_band).toFloat().clip(WORK_REGION)
)

TASKS.append(exports.image_to_drive(
    _total_obs, product="obs_count", aoi="district", params=params,
    region=WORK_REGION.bounds(), band_order=[_obs_band],
    scale_m=FIT_SCALE, suffix=OBS_SOURCE,
))
print("submitted:", TASKS[-1].status().get("description"))

In [ ]:
# COLAB: RUN THIS CELL
# --- 3. UTFVI six-class image per epoch, one multi-band raster ------------
# Built on the SAME source and scale Phase 3 used, so figure 4 and the Phase 3
# class-share series describe the same product.
_utfvi_scenes = uhi_metrics.source_collection(
    params["spatial_stats"]["epochs_source"], params, region=WORK_REGION
)
_class_band = params["uhi"]["utfvi"]["class_band_name"]

_layers = []
for _epoch in EPOCHS:
    _composite = uhi_metrics.epoch_composite(
        params["spatial_stats"]["epochs_source"], params, _epoch,
        collection=_utfvi_scenes,
    )
    _index = uhi_metrics.utfvi(
        _composite, params, WORK_REGION, scale_m=UTFVI_SCALE_M
    )
    _layers.append(
        uhi_metrics.utfvi_class_image(_index, params).rename(f"{_class_band}_{_epoch}")
    )
    print(f"  built {_epoch}")

UTFVI_BANDS = [f"{_class_band}_{_epoch}" for _epoch in EPOCHS]
TASKS.append(exports.image_to_drive(
    ee.Image.cat(_layers).toFloat().clip(WORK_REGION),
    product="utfvi_class", aoi="district", params=params,
    region=WORK_REGION.bounds(), band_order=UTFVI_BANDS,
    scale_m=UTFVI_SCALE_M, suffix="epochs",
))
print("submitted:", TASKS[-1].status().get("description"))
print()
print(f"{len(TASKS)} task(s) queued to Drive folder "
      f"'{params['exports']['drive_folder']}'.")
print("Re-run the NEXT cell until every state reads COMPLETED.")

In [ ]:
# COLAB: RUN THIS CELL  (re-runnable - poll until every state reads COMPLETED)
display(exports.describe_tasks(TASKS))

## Step 3 - the per-epoch driver sample (figure 7)

No batch task: `uhi_metrics.sample_drivers` returns a frame directly. One call
per year, pooled and tagged with its epoch. Roughly one to three minutes.

The sample is drawn at `uhi.drivers.sample_scale_m`, deliberately coarser than
the 30 m grid, because adjacent 30 m LST pixels are near-duplicates and sampling
at 30 m buys spatial autocorrelation rather than information. The seed is fixed,
so a re-run reproduces the same points and the same fitted slopes.

In [ ]:
# COLAB: RUN THIS CELL
DRIVER_SOURCE = params["spatial_stats"]["epochs_source"]
_driver_scenes = uhi_metrics.source_collection(
    DRIVER_SOURCE, params, region=WORK_REGION
)

_frames = []
for _epoch in EPOCHS:
    _start, _end = uhi_metrics.epoch_years(params, _epoch)
    for _year in range(_start, _end + 1):
        try:
            _frame = uhi_metrics.sample_drivers(
                DRIVER_SOURCE, params, _year, WORK_REGION,
                collection=_driver_scenes,
            )
        except Exception as _error:  # noqa: BLE001 - a year with no usable scene
            print(f"  {_year}: skipped ({type(_error).__name__})")
            continue
        if _frame.empty:
            print(f"  {_year}: no rows")
            continue
        _frame["epoch"] = _epoch
        _frames.append(_frame)
    print(f"{_epoch}: {sum(len(f) for f in _frames):,} rows so far")

DRIVER_SAMPLES = pd.concat(_frames, ignore_index=True)
DRIVER_CSV = f"{OUTPUT_DIR}/driver_samples_by_epoch.csv"
DRIVER_SAMPLES.to_csv(DRIVER_CSV, index=False)
print()
print(f"{len(DRIVER_SAMPLES):,} sampled pixels -> {DRIVER_CSV}")
display(DRIVER_SAMPLES.groupby("epoch").size().rename("rows").to_frame())
print()
print("CAVEAT: sampled pixels are spatially autocorrelated, so the OLS standard")
print("errors on figure 7 are too small. Read the slopes as a screening result.")

---

# WAIT HERE

Poll the task cell above until **every** state reads `COMPLETED`, then copy the
new `.tif` files from your Drive export folder into `data/interim/`.

```
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/colombo_uhi_exports/*.tif data/interim/
```

**Also copy the Phase 4 rasters** if this is a fresh runtime - figures 1 and 2
reuse them and Phase 8 does not re-export them:

* `lst_trend_district_*_100m_landsat_oli_dry.tif`
* `lst_decadal_district_*_100m_landsat_dry.tif`

---

# Part 2 - render

## Step 4 - discovery, before anything is drawn

Every input is located and reported **first**. A render loop that discovers a
missing raster on figure 9 has already spent minutes drawing eight others and
reports the problem as a traceback; this names every gap at once.

In [ ]:
# COLAB: RUN THIS CELL
def _tif(product: str, suffix: str, res_m: int) -> "str | None":
    # Path to an exported raster in data/interim, or None if it is absent.
    name = exports.export_name(product, "district", params, res_m=res_m, suffix=suffix)
    path = f"{INTERIM_DIR}/{name}.tif"
    return path if os.path.exists(path) else None


def _csv(name: str) -> "str | None":
    path = f"{OUTPUT_DIR}/{name}"
    return path if os.path.exists(path) else None


TREND_TIF = _tif("lst_trend", params["trends"]["pixel_sources"][0], FIT_SCALE)
DECADAL_LANDSAT_TIF = _tif("lst_decadal", DECADAL_LANDSAT_SOURCE, FIT_SCALE)
DECADAL_MODIS_TIF = _tif("lst_decadal", DECADAL_MODIS_SOURCE, MODIS_SCALE_M)
OBS_TIF = _tif("obs_count", OBS_SOURCE, FIT_SCALE)
UTFVI_TIF = _tif("utfvi_class", "epochs", UTFVI_SCALE_M)

# Figure 2 wants the FDR-CORRECTED raster. That correction is pure Python on the
# exported p-values, so it is redone here rather than re-exported - which also
# means figure 2 cannot silently use a stale FDR run from another session.
TREND_FDR_TIF = None
if TREND_TIF:
    _method = params["trends"]["fdr"]["method"]
    TREND_FDR_TIF = f"{INTERIM_DIR}/{os.path.basename(TREND_TIF)[:-4]}_fdr_{_method}.tif"
    _fdr_summary = trends.apply_fdr_to_raster(
        TREND_TIF, TREND_FDR_TIF, params, method=_method
    )
    print(f"FDR ({_method}): {_fdr_summary['n_significant']:,} of "
          f"{_fdr_summary['n_tested']:,} tested pixels significant "
          f"({_fdr_summary['fraction_of_tested']:.1%})")
    print()

AVAILABLE = {
    "decadal_landsat_tif": DECADAL_LANDSAT_TIF,
    "decadal_modis_tif": DECADAL_MODIS_TIF,
    "sensor_offsets_csv": _csv("sensor_offsets_cmc.csv"),
    "trend_fdr_tif": TREND_FDR_TIF,
    "obs_count_tif": OBS_TIF,
    "utfvi_class_tif": UTFVI_TIF,
    "zones_geojson": _csv("gn_divisions_colombo.geojson"),
    "gi_star_csv": _csv("gi_star_gn_2020s.csv"),
    "ehsa_csv": _csv("ehsa_gn_landsat_oli_dry.csv"),
    "suhii_csv": _csv("suhii_2000_2025.csv"),
    "driver_samples_csv": _csv("driver_samples_by_epoch.csv"),
    "gwr_csv": _csv("gwr_local_coefficients_gn_2020s.csv"),
    "counterfactual_baseline_tif": _csv("greening_counterfactual_baseline.tif"),
    "counterfactual_greened_tif": _csv("greening_counterfactual_greened.tif"),
    "counterfactual_delta_tif": _csv("greening_counterfactual_delta.tif"),
    "priority_csv": _csv("greening_priority_gn.csv"),
}

print("INPUTS")
for _name, _path in AVAILABLE.items():
    print(f"  {'OK  ' if _path else 'MISS'}  {_name:<30} {_path or ''}")

GAPS = viz.missing_inputs(AVAILABLE, params)
print()
if GAPS:
    print("FIGURES THAT CANNOT BE DRAWN:")
    for _index, _names in sorted(GAPS.items()):
        print(f"  figure {_index:>2}: missing {', '.join(_names)}")
    print()
    print("Copy the missing rasters from Drive into data/interim/ (see WAIT")
    print("HERE above), then re-run this cell. Figures that CAN be drawn will")
    print("still be drawn below; the rest will say so.")
else:
    print("All eleven figures can be drawn.")

In [ ]:
# COLAB: RUN THIS CELL
# Shared readers, so no figure cell reopens a file its own way.
ZONES = None
if AVAILABLE["zones_geojson"]:
    ZONES = gpd.read_file(AVAILABLE["zones_geojson"]).to_crs(
        params["crs"]["analysis_epsg"]
    )
    print(f"{len(ZONES)} zones in {ZONES.crs}")


def _surface(path):
    # Read a single-band float GeoTIFF, nodata as NaN.
    import rasterio

    with rasterio.open(path) as handle:
        array = handle.read(1).astype("float64")
        if handle.nodata is not None:
            array = np.where(array == handle.nodata, np.nan, array)
    return array


WRITTEN = {}


def _emit(index: int, figure) -> None:
    # Save one report figure at report.dpi and show it.
    entry = next(e for e in viz.report_manifest(params) if e["index"] == index)
    path = viz.save_report_figure(figure, params, index, entry["slug"])
    WRITTEN[index] = str(path)
    print(f"figure {index}: {path}  ({os.path.getsize(path) / 1024:.0f} KB)")
    display(Image(filename=str(path)))

### Figure 1 - decadal mean dry-season LST

Two rows on purpose. Read the **top row for geography** and the **bottom row for
level**. If the two rows step between decades in the same direction and by
similar amounts, the cross-sensor finding is wrong - check that before quoting
anything else in this notebook.

In [ ]:
# COLAB: RUN THIS CELL
if 1 in GAPS:
    print("figure 1 skipped; missing", ", ".join(GAPS[1]))
else:
    _order = trends.decadal_band_order(params)
    _landsat, _ = trends.read_trend_raster(
        AVAILABLE["decadal_landsat_tif"], params, band_order=_order
    )
    _modis, _ = trends.read_trend_raster(
        AVAILABLE["decadal_modis_tif"], params, band_order=_order
    )
    _offsets = pd.read_csv(AVAILABLE["sensor_offsets_csv"])

    # Check the shared stretch before drawing: a stretch that clips understates
    # the extremes it exists to show.
    _vis = params["report"]["decadal"]["shared_vis"]
    for _label, _arrays in (("landsat", _landsat), ("modis", _modis)):
        for _decade in [d for d, _, _ in trends.resolve_decades(None, params)]:
            _clipped = viz.saturated_fraction(
                _arrays[f"mean_{_decade}"], _vis["min"], _vis["max"]
            )
            _flag = "  <-- WIDEN report.decadal.shared_vis" if (
                _clipped > params["report"]["max_saturated_fraction"]
            ) else ""
            print(f"  {_label:<8} {_decade}: {_clipped:.2%} outside the stretch{_flag}")

    _emit(1, viz.build_decadal_lst_panel_figure(
        _landsat, _modis, params, sensor_offsets=_offsets,
    ))

### Figure 2 - Sen's slope with FDR-significance stippling

One map, not two: stippling puts the rate and its confidence in the same place,
so a reader cannot take a rate off one panel and its confidence off another.
Grey is **not** "no trend" - it is a pixel that was never tested. Figure 3 is
what says why.

In [ ]:
# COLAB: RUN THIS CELL
if 2 in GAPS:
    print("figure 2 skipped; missing", ", ".join(GAPS[2]))
else:
    _arrays, _ = trends.read_trend_raster(
        AVAILABLE["trend_fdr_tif"], params,
        band_order=["sen_slope", "sen_slope_fdr", "p_two_sided",
                    "p_adjusted", "significant", "n_years"],
    )
    _source = params["trends"]["pixel_sources"][0]
    _emit(2, viz.build_sen_slope_stipple_figure(
        _arrays, params,
        title=f"Sen's slope of land surface temperature, {_source} ({FIT_SCALE} m)",
    ))

### Figure 3 - per-pixel valid-observation count

`CLAUDE.md` caveat 2, as a product rather than a footnote. The cumulative panel
is the point: a colour ramp cannot be integrated by eye, and "most of the city
is thin" and "one corner is thin" look alike on a map.

In [ ]:
# COLAB: RUN THIS CELL
if 3 in GAPS:
    print("figure 3 skipped; missing", ", ".join(GAPS[3]))
else:
    _obs, _ = trends.read_trend_raster(
        AVAILABLE["obs_count_tif"], params,
        band_order=[params["composites"]["obs_count_band"]],
    )
    _counts = _obs[params["composites"]["obs_count_band"]]
    _finite = _counts[np.isfinite(_counts)]
    print("Percentiles of usable dry-season observations per pixel:")
    for _q in (1, 5, 25, 50, 75, 95, 99):
        print(f"  p{_q:<3d} {np.percentile(_finite, _q):6.0f}")
    _vis = params["report"]["obs_count_vis"]
    _clipped = viz.saturated_fraction(_counts, _vis["min"], _vis["max"])
    print(f"\nOutside the {_vis['min']}-{_vis['max']} stretch: {_clipped:.2%}")
    if _clipped > params["report"]["max_saturated_fraction"]:
        print("  -> widen report.obs_count_vis and re-run this cell.")

    _emit(3, viz.build_obs_count_figure(
        _obs, params,
        title=f"Valid dry-season observations per pixel, {OBS_SOURCE}, "
              f"{params['time']['start_year']}-{params['time']['end_year']}",
    ))

### Figure 4 - UTFVI six-class maps by epoch

UTFVI is referenced to **each epoch's own mean**, so a difference between these
panels is a redistribution of heat within that epoch and **never** evidence of
warming. Figure 2 is what measures warming.

In [ ]:
# COLAB: RUN THIS CELL
if 4 in GAPS:
    print("figure 4 skipped; missing", ", ".join(GAPS[4]))
else:
    _class_band = params["uhi"]["utfvi"]["class_band_name"]
    _bands = [f"{_class_band}_{_epoch}" for _epoch in EPOCHS]
    _raster, _ = trends.read_trend_raster(
        AVAILABLE["utfvi_class_tif"], params, band_order=_bands
    )
    EPOCH_CLASSES = {_epoch: _raster[_band] for _epoch, _band in zip(EPOCHS, _bands)}
    _emit(4, viz.build_utfvi_epoch_maps_figure(
        EPOCH_CLASSES, params,
        title=f"UTFVI by epoch, {params['spatial_stats']['epochs_source']} "
              f"({UTFVI_SCALE_M} m)",
    ))

### Figure 5 - Getis-Ord Gi* and emerging hot spots

Both are **within-epoch** statistics, so a spatially uniform sensor step cancels
and the cluster geography is valid - but no epoch-to-epoch temperature magnitude
may be read off them (`caveats.within_epoch_only`).

The EHSA palette is the one the Phase 8 colour check could not fix: seventeen
categories cannot be separated by colour even for a reader with normal colour
vision. Read it from the ordered legend.

In [ ]:
# COLAB: RUN THIS CELL
if 5 in GAPS:
    print("figure 5 skipped; missing", ", ".join(GAPS[5]))
else:
    _gi = pd.read_csv(AVAILABLE["gi_star_csv"])
    _emit(5, viz.build_hotspot_map_figure(
        ZONES, _gi, params,
        title="Getis-Ord Gi* by GN division, 2020s epoch",
    ))

    _ehsa = pd.read_csv(AVAILABLE["ehsa_csv"])
    _figure = viz.build_ehsa_map_figure(
        ZONES, _ehsa, params,
        title="Emerging hot spot analysis by GN division",
    )
    _path = viz.save_report_figure(_figure, params, 5, "hotspots_ehsa")
    print("figure 5b:", _path)
    display(Image(filename=str(_path)))

### Figure 6 - SUHII by source and rural definition

Landsat, MODIS Terra day, Terra night, Aqua day and Aqua night, each under
**both** rural definitions. Night-time UHI comes only from MODIS: Landsat sees
one ~10:30 overpass.

SUHII is the one product the cross-sensor step does **not** damage, because it
is a within-year urban-minus-rural difference and a common-mode offset cancels.

In [ ]:
# COLAB: RUN THIS CELL
if 6 in GAPS:
    print("figure 6 skipped; missing", ", ".join(GAPS[6]))
else:
    SUHII = pd.read_csv(AVAILABLE["suhii_csv"])
    print("sources:", sorted(SUHII["source"].unique()))
    print("rural definitions:", sorted(SUHII["rural_definition"].unique()))
    _emit(6, viz.build_suhii_figure(SUHII, params))

### Figure 7 - LST against NDVI and NDBI, faceted by epoch

Rows share axes so the **slopes** are comparable by eye. They are not comparable
in **level**: the epochs pool different Landsat sensors, so a vertical shift
between rows is not a measurement of warming.

In [ ]:
# COLAB: RUN THIS CELL
if 7 in GAPS:
    print("figure 7 skipped; missing", ", ".join(GAPS[7]))
else:
    _samples = pd.read_csv(AVAILABLE["driver_samples_csv"])
    _emit(7, viz.build_lst_vs_index_facet_figure(_samples, params))

### Figure 8 - GWR local coefficients

The end of the escalation path: OLS, residual Moran's I, spatial lag/error, then
geographically weighted regression. These coefficients describe **GN polygons**,
not pixels and not people (`caveats.zonal_not_pixel`).

In [ ]:
# COLAB: RUN THIS CELL
if 8 in GAPS:
    print("figure 8 skipped; missing", ", ".join(GAPS[8]))
else:
    _gwr = pd.read_csv(AVAILABLE["gwr_csv"])
    _emit(8, viz.build_gwr_coefficient_figure(
        ZONES, _gwr, params,
        title="GWR local coefficients by GN division, 2020s epoch",
    ))

### Figure 9 - greening counterfactual

**Not a forecast.** This is a counterfactual on *observed* predictors - "if
these zones were greened today" - which is why it can be mapped at all: it rests
on the validated random forest alone, with no land-cover projection underneath
it. The 2030 and 2036 horizons do rest on the CA-Markov component, which did not
validate, so no map of them exists.

The validation metrics are read from the **GeoTIFF's own tags**, stamped there
by the guard that allowed the file to be written. `validation_reports.json` does
not carry the `lst_scenario` entry - a gap in notebook 06 recorded in
`PROGRESS.md` - so the tag is the authoritative copy.

The district-wide mean difference is **not** the result: it is diluted by every
cell nothing was done to. The priority-zone mean is what the report quotes.

In [ ]:
# COLAB: RUN THIS CELL
if 9 in GAPS:
    print("figure 9 skipped; missing", ", ".join(GAPS[9]))
else:
    _baseline = _surface(AVAILABLE["counterfactual_baseline_tif"])
    _greened = _surface(AVAILABLE["counterfactual_greened_tif"])
    _delta = _surface(AVAILABLE["counterfactual_delta_tif"])
    _report = reporting.scenario_report_from_raster(
        AVAILABLE["counterfactual_delta_tif"], params
    )
    # The greened cells are exactly those the lever moved.
    _mask = np.isfinite(_delta) & (np.abs(_delta) > 1e-9)
    print(f"greened cells: {int(_mask.sum()):,}  "
          f"mean difference inside them: {np.nanmean(_delta[_mask]):+.2f} degC")
    print(f"district-wide mean: {np.nanmean(_delta):+.2f} degC  <-- NOT the result")

    _emit(9, viz.build_scenario_triptych_figure(
        _baseline, _greened, _delta, params,
        report=_report, priority_mask=_mask,
    ))

### Figure 10 - greening priority, with the top zones labelled

Ten labels, not sixty: sixty over the CMC core is an unreadable mat. The full
ranked list is `greening_priority_top_gn.csv`.

Two things travel with this map. The **weights are judgements**, and the
**criteria are near-collinear over Colombo** - the leave-one-out ablation, not
the consistency ratio, is what says how much the method adds.

In [ ]:
# COLAB: RUN THIS CELL
if 10 in GAPS:
    print("figure 10 skipped; missing", ", ".join(GAPS[10]))
else:
    PRIORITY = pd.read_csv(AVAILABLE["priority_csv"])
    _meta_path = f"{OUTPUT_DIR}/greening_priority_gn_meta.json"
    _ahp = None
    if os.path.exists(_meta_path):
        with open(_meta_path, encoding="utf-8") as _fh:
            _ahp = json.load(_fh).get("ahp")

    _rank_column = "rank_ahp" if "rank_ahp" in PRIORITY.columns else "rank"
    print(f"ranking column: {_rank_column}; {len(PRIORITY)} zones")
    print(PRIORITY.nsmallest(params["report"]["label_top_n"], _rank_column)
          .head(10).to_string(index=False))

    _figure = viz.build_greening_priority_map_figure(
        ZONES, PRIORITY, params, _ahp,
        label_top_n=int(params["report"]["label_top_n"]),
    )
    _emit(10, _figure)

### Figure 11 - data provenance

Generated from `config/params.yaml`, not typed, so it cannot disagree with the
code. Sources shown in grey are configured but referenced by no analysis step -
"configured" and "used" are different claims.

In [ ]:
# COLAB: RUN THIS CELL
PROVENANCE = reporting.provenance_frame(params)
_unused = PROVENANCE[PROVENANCE["used_by"].str.startswith("not ")]
print(f"{len(PROVENANCE)} sources; {len(_unused)} referenced by nothing:")
for _key in _unused["key"]:
    print("  ", _key)
print()
_emit(11, viz.build_provenance_table_figure(PROVENANCE, params))

---

# Part 3 - the colour check, the generated docs, and the bundle

## Step 5 - the colour-vision check, measured

Two different tests, because one cannot judge both kinds of palette.
**Categorical** palettes are judged on the minimum pairwise CIE76 difference
under each simulated deficiency; **ramps** are judged on whether L* stays
monotonic. Judging a ramp by pairwise difference is the mistake the split
avoids - adjacent stops of a ramp are meant to be close.

Two palettes are exempt, with their measured numbers recorded. Every other
palette must pass.

In [ ]:
# COLAB: RUN THIS CELL
CVD = viz.cvd_report(params)
display(CVD[["path", "kind", "metric", "worst", "threshold", "passed",
             "exempt", "worst_pair", "verdict"]])

_cvd_csv = f"{OUTPUT_DIR}/palette_cvd_check.csv"
CVD.to_csv(_cvd_csv, index=False)
print("Wrote", _cvd_csv)
print()

FAILURES = viz.cvd_failures(CVD)
if len(FAILURES):
    print("*** PALETTES THAT FAIL AND ARE NOT EXEMPT ***")
    for _, _row in FAILURES.iterrows():
        print(f"  {_row['path']}: {_row['verdict']}")
    print()
    print("Fix them in params.yaml, record the before/after numbers beside the")
    print("palette, and re-run Part 2. Do NOT exempt one to make this pass.")
else:
    print("Every non-exempt palette passes.")
    print()
    print("Exempt, with the reason:")
    for _, _row in CVD[CVD["exempt"]].iterrows():
        print(f"  {_row['path']}: {_row['verdict']}")
        print(f"      {_row['reason']}")

## Step 6 - regenerate the methods and limitations sections

Both documents are written from `params.yaml` and from the committed products.
`tests/test_reporting.py` fails if the committed copy disagrees with what the
code produces, so this cell is what keeps that test green after any config
change.

In [ ]:
# COLAB: RUN THIS CELL
DOCS = reporting.write_docs(params)
for _name, _path in DOCS.items():
    print(f"{_name:<12} -> {_path}  ({os.path.getsize(_path) / 1024:.1f} KB)")
print()
print("=" * 72)
print(reporting.limitations_markdown(params).split("## Part 2")[1][:1800])

## Step 7 - BRING THE RESULTS HOME

**Do not skip this.** Everything Part 2 produced lives in a Colab VM that will
be recycled. The GeoTIFFs in `data/interim/` are deliberately excluded - they
are tens of MB and fully reproducible from the Drive exports. The figures, the
generated docs and the CSVs are not reproducible without another full run.

In [ ]:
# COLAB: RUN THIS CELL
_bundle = "/content/phase8_outputs.zip"
_written = []
with zipfile.ZipFile(_bundle, "w", zipfile.ZIP_DEFLATED) as _zip:
    for _folder in (FIGURE_DIR, OUTPUT_DIR, "docs"):
        for _root, _, _files in os.walk(_folder):
            for _name in _files:
                if _name == ".gitkeep":
                    continue
                _path = os.path.join(_root, _name)
                _zip.write(_path, _path)
                _written.append(_path)

print(f"{len(_written)} file(s) bundled into {_bundle}")
print()
print(f"Report figures written this run ({len(WRITTEN)} of 11):")
for _index in sorted(WRITTEN):
    print(f"  {_index:>2}. {WRITTEN[_index]}")
if len(WRITTEN) < 11:
    print()
    print(f"*** {11 - len(WRITTEN)} figure(s) were NOT drawn. See Step 4. ***")
print()
print("Download it from the Colab Files pane (folder icon, left sidebar),")
print("unzip into the repo root locally, then:")
print("    git add figures/report data/outputs docs && git commit && git push")
try:
    from google.colab import files
    files.download(_bundle)
except Exception as _error:
    print()
    print(f"(auto-download unavailable: {_error} - use the Files pane)")

## What to check before signing Phase 8 off

| # | Check | Why it matters |
|---|---|---|
| 1 | Part 1 submitted exactly **three** tasks and all reached `COMPLETED` | Phase 8 adds three exports; more means something re-ran |
| 2 | Step 4 reported **no** missing inputs | Every gap is named there, before anything is drawn |
| 3 | **Eleven** PNGs in `figures/report/`, each at 300 dpi | `PIL.Image.open(p).info["dpi"]` should read `(300, 300)` |
| 4 | On figure 1, the **Landsat row steps between decades and the MODIS row does not** | If both step together the cross-sensor finding is wrong, and every trend product in this project rests on it |
| 5 | No panel on figure 1 or 3 exceeds `report.max_saturated_fraction` | A clipped stretch understates the extremes the figure exists to show |
| 6 | The colour table has **zero non-exempt failures** | Fix the palette, never the exemption |
| 7 | Figure 9's priority-zone mean reads about **-0.84 degC**, not the district-wide figure | The district-wide mean is diluted by every cell nothing was done to |
| 8 | Figure 10 labels ten divisions and the leader lines do not cross | Labels are ordered by centroid latitude; crossing lines mean the spread failed |
| 9 | `python -m pytest tests/ -q` still passes **after** Step 6 rewrote the docs | The drift test compares the committed docs against freshly generated ones |
| 10 | Read `docs/methods.md` once, end to end | Generated prose can be numerically correct and still unreadable; that is the one thing no test can check |

### Then

```bash
git add figures/report data/outputs docs config/params.yaml
git commit -m "Phase 8: report figures, generated methods and limitations"
```